In [ ]:
import pandas as pd
import numpy as np
import os
import gc

import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# set a seed value
torch.manual_seed(555)

from sklearn.utils import shuffle
from sklearn.metrics import roc_auc_score, accuracy_score

import transformers
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification
from transformers import AdamW

import warnings
warnings.filterwarnings("ignore")


print(torch.__version__)

2.6.0+cu124


## Introduction

This notebook is divided into 3 Sections and an Appendix. In section 1 we will look at how to format input data for Bert and XLM-Roberta and review the ouput that these models produce. In section 2 we will load the competition data and create 5 folds. In section 3 we will fine-tune a 3 fold cv Bert model and a single fold XLM-RoBERTa model - using Pytorch with a single xla device (TPU). Finally, in the Appendix I've included some info that I'm finding helpful as I learn how to use pre-trained transformer models.

## Contents

<a href='#Section_1'>Section 1</a><br>
<a href='#BERT'>1.1. Explore BERT</a><br>
<a href='#XLM-Roberta'>1.2. Explore XLM-RoBERTa</a><br>
<a href='#Manual_formatting_of_model_input_data'>1.3. Manual formatting of model input data</a><br>
<a href='#Overflowing_tokens_and_Stride'>1.4. Overflowing tokens and Stride</a><br>

<a href='#Section_2'>Section 2</a><br>
<a href='#Load_the_Data'>2.1. Load the Data</a><br>
<a href='#Create_5_Folds'>2.2. Create 5 Folds</a><br>

<a href='#Section_3'>Section 3</a><br>
<a href='#Train_a_Bert_Model'>3.1. Train a BERT Model</a><br>
<a href='#Train_an_XLM-Roberta_Model'>3.2. Train an XLM-RoBERTa Model </a><br>

<a href='#Appendix'>Appendix</a><br>
<a href='#Acronyms'>A1 - Acronyms</a><br>
<a href='#GLUE_Datasets'>A2 - GLUE Datasets</a><br>
<a href='#Datasets_Separated_by_Task'>A3 - Datasets Separated by Task</a><br>
<a href='#Papers'>A4 - Papers</a><br>
<a href='#NLP_Applications'>A5 - What is NLP used for?</a><br>
<a href='#Helpful_Resources'>A6 - Helpful Resources</a><br>



| <a id='Section_1'></a>

# Section 1

In this section we'll look at what input the Bert and XLM-RoBERTa models expect and what output they produce. We will also use a tokenizer to automatically process one sentence and a pair of sentences into the correct input format for each model.

## 1.1. Explore BERT

BERT is a pre-trained language model that can be fine tuned to perform NLP tasks. BERT stands for Bidirectional Encoder Representations from Transformers. Bidirectional means that the model is able to read text from both left-to-right and from right-to-left. This capability helps it to understand context. BERT's other super-power is that it can understand 100 languages.

You can access BERT and other pre-trained models through a library called [Transformers](https://github.com/huggingface/transformers). The team at Hugging Face created this library. It contains many transformer models and tokenizers that use a common interface. The Transformers library is available in Kaggle notebooks by default - simply type: import Transformers.

Several pre-trained BERT models are available - different sizes, monolingual and multilingual. These are a few types:

- bert-base-uncased
- bert-base-cased
- bert-large-uncased
- bert-large-cased
- bert-base-multilingual-uncased
- bert-base-multilingual-cased

*uncased* - All text was converted to lower case before training the model.<br>
*cased* - The text used to train the model was not converted to lower case.

This is a link to a full searchable listing of all model types:<br>
https://huggingface.co/models

This is a link to the BERT paper:<br>
https://arxiv.org/pdf/1810.04805.pdf


## 1.2. Explore XLM-RoBERTa

XLM means Cross-lingual Language Model. XLM-RoBERTa (XLM-R) is a pre-trained multilingual model that outperforms multiligual BERT. One reason for this is that XLM-R was trained using a lot more data. XLM-R was also trained on 100 languages.

Several versions of xlm roberta are available in the Transformers library. Here are two:

- xlm-roberta-base
- xlm-roberta-large

This is the link to the XLM-RoBERTa paper:<br>
https://arxiv.org/pdf/1911.02116.pdf

## XLM-RoBERTa Vocabulary

In [ ]:
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification

MODEL_TYPE = 'xlm-roberta-base'

tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_TYPE)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

In [ ]:
# Check the vocab size

tokenizer.vocab_size

250002

In [ ]:
# What are the special tokens

tokenizer.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '</s>',
 'unk_token': '<unk>',
 'sep_token': '</s>',
 'pad_token': '<pad>',
 'cls_token': '<s>',
 'mask_token': '<mask>'}

In [ ]:
print('bos_token_id <s>:', tokenizer.bos_token_id)
print('eos_token_id </s>:', tokenizer.eos_token_id)
print('sep_token_id </s>:', tokenizer.sep_token_id)
print('pad_token_id <pad>:', tokenizer.pad_token_id)

bos_token_id <s>: 0
eos_token_id </s>: 2
sep_token_id </s>: 2
pad_token_id <pad>: 1


## What input does XLM-RoBERTa expect?

In [ ]:
from transformers import XLMRobertaForSequenceClassification

MODEL_TYPE = 'xlm-roberta-base'

model = XLMRobertaForSequenceClassification.from_pretrained(
                 MODEL_TYPE,
                 num_labels = 3 # The number of output labels. 2 for binary classification.
              )

# Define and initialize input tensors with sample data
# Replace these with your actual data
b_input_ids = torch.tensor([[0, 35378, 2685, 5, 2, 1, 1, 1, 1, 1]])
b_input_mask = torch.tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])
b_labels = torch.tensor([0])  # Replace with your actual label

outputs = model(input_ids=b_input_ids,
                 attention_mask=b_input_mask,
                 labels=b_labels)



# These are the model inputs:
#   input_ids (type: torch tensor)
#   attention_mask (type: torch tensor)
#   labels (type: torch tensor)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 1. input_ids
# -------------

# The input_ids are the sentence or sentences represented as tokens.
# These are special tokens:

# bos_token_id <s>: 0
# eos_token_id </s>: 2
# sep_token_id </s>: 2
# pad_token_id <pad>: 1


# XLM-RoBERTa expects every row in the input_ids to have the special tokens included as follows:

# For one sentence as input:
# <s> ...word tokens... </s>

# For two sentences as input:<br>
# <s> ...sentence1 tokens... </s></s>..sentence2 tokens... </s>


# This is an example of an encoded sentence with padding (pad token value: 1).

# [0, 35378, 2685, 5, 2, 1, 1, 1, 1, 1]




# 2. token_type_ids
# ------------------

# XLM-RoBERTa does not use token_type_ids like BERT does.
# Therefore, there's no need to create token_type_ids.



# 3. attention_mask
# ------------------

# The attention mask has the same length as the input_ids.
# It tells the model which tokens in the input_ids are works and which are padding.
# 1 indicates a word (or special token) and 0 indicates padding.

# For example, the attention mask for the above input_ids is as follows:
# [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]


# 3. labels
# ----------

# The label (target) for each row in the input_ids.
# The labels are integers representing each target class e.g. 1, 2, 3 etc.

# For example if we have three target classes (0, 1 and 2) then
# the labels could look like this for a batch size of 8:

# [0, 2, 0, 1, 2, 0, 3, 1]

## How to use a tokenizer to create XLM-RoBERTa input

### For one input sentence

In [ ]:
MAX_LEN = 10 # This value could be set as 256, 512 etc.

sentence1 = 'Hello there.'

encoded_dict = tokenizer.encode_plus(
            sentence1,
            add_special_tokens = True,
            max_length = MAX_LEN,
            pad_to_max_length = True,
            return_attention_mask = True,
            return_tensors = 'pt', # return pytorch tensors
            truncation=True  # Add truncation
       )

encoded_dict

{'input_ids': tensor([[    0, 35378,  2685,     5,     2,     1,     1,     1,     1,     1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])}

In [ ]:
# These have already been converted to torch tensors.
input_ids = encoded_dict['input_ids'][0]
att_mask = encoded_dict['attention_mask'][0]

print(input_ids)
print(att_mask)

tensor([    0, 35378,  2685,     5,     2,     1,     1,     1,     1,     1])
tensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])


### For two input sentences

In [ ]:
MAX_LEN = 15

sentence1 = 'Hello there.'
sentence2 = 'How are you?'

encoded_dict = tokenizer.encode_plus(
            sentence1, sentence2,
            add_special_tokens = True,
            max_length = MAX_LEN,
            pad_to_max_length = True,
            return_attention_mask = True,
            return_tensors = 'pt' # return pytorch tensors
       )


encoded_dict

{'input_ids': tensor([[    0, 35378,  2685,     5,     2,     2, 11249,   621,   398,    32,
             2,     1,     1,     1,     1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])}

In [ ]:
input_ids = encoded_dict['input_ids'][0]
att_mask = encoded_dict['attention_mask'][0]

# These are torch tensors.
print(input_ids)
print(att_mask)

tensor([    0, 35378,  2685,     5,     2,     2, 11249,   621,   398,    32,
            2,     1,     1,     1,     1])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0])


## Decoding a sequence of tokens

In [ ]:
# input_ids from above

input_ids = encoded_dict['input_ids'][0]

print(input_ids)

tensor([    0, 35378,  2685,     5,     2,     2, 11249,   621,   398,    32,
            2,     1,     1,     1,     1])


In [ ]:
# https://huggingface.co/transformers/main_classes/tokenizer.html
# skip_special_tokens – if set to True, will replace special tokens.

a = tokenizer.decode(input_ids,
                skip_special_tokens=False)

b = tokenizer.decode(input_ids,
                skip_special_tokens=True)



print(a)
print(b)

<s> Hello there.</s></s> How are you?</s><pad><pad><pad><pad>
Hello there. How are you?


| <a id='Manual_formatting_of_model_input_data'></a>

## 1.3. Manual formatting of model input data

So far we've used tokenizer.encode_plus to automatically format input data. I've included the following resource links because there are things that one needs to be aware of when writing code for manual formatting. For example XLM-RoBERTa uses a SentencePiece-based tokenizer but BERT does not. It's also good to know what SentencePiece tokenization is and how it works.

Hugging Face Tokenizer docs<br>
https://huggingface.co/transformers/main_classes/tokenizer.html

Abhishek Thakur<br>
Data Processing For Question & Answering Systems: BERT vs. RoBERTa<br>
(Note that this video covers RoBERTa and not XLM-RoBERTa)<br>
https://www.youtube.com/watch?v=6a6L_9USZxg

Abhishek Thakur<br>
Sentencepiece Tokenizer With Offsets For T5, ALBERT, XLM-RoBERTa And Many More<br>
https://youtu.be/U51ranzJBpY

SentencePiece Paper<br>
https://arxiv.org/abs/1808.06226

SentencePiece Github<br>
https://github.com/google/sentencepiece

The following Hugging face models use a SentencePiece-based tokenizer:<br>
T5, ALBERT, CamemBERT, XLMRoBERTa and XLNet

| <a id='Overflowing_tokens_and_Stride'></a>

## 1.4. Overflowing tokens and Stride

When a sentence is truncated (because it's length exceeds max_length) it's possible to get the tokenizer to return the tokens that were cut off. These truncated tokens will be returned in a list called overflowing_tokens.

In [ ]:
MAX_LEN = 15 # This value could be set as 256, 512 etc.

sentence1 = 'Hello there. How are you? Have a nice day. This is a test?'


encoded_dict = tokenizer.encode_plus(
            sentence1,
            max_length = MAX_LEN,
            stride=0,
            pad_to_max_length = True,
            return_overflowing_tokens=True,
       )


encoded_dict

{'overflowing_tokens': [83, 10, 3034, 32], 'num_truncated_tokens': 4, 'input_ids': [0, 35378, 2685, 5, 11249, 621, 398, 32, 31901, 10, 26267, 5155, 5, 3293, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Above we see that 4 tokens were truncated. The token numbers are shown in the list called overflowing_tokens.

We can also specify a stride. This adds tokens to the front of the 'overflowing_tokens list creating an overlap. The best way to illustrate this is with an example:

In [ ]:
MAX_LEN = 15 # This value could be set as 256, 512 etc.

sentence1 = 'Hello there. How are you? Have a nice day. This is a test?'


encoded_dict = tokenizer.encode_plus(
            sentence1,
            max_length = MAX_LEN,
            stride=3,
            pad_to_max_length = True,
            return_overflowing_tokens=True,
       )


encoded_dict

{'overflowing_tokens': [5155, 5, 3293, 83, 10, 3034, 32], 'num_truncated_tokens': 4, 'input_ids': [0, 35378, 2685, 5, 11249, 621, 398, 32, 31901, 10, 26267, 5155, 5, 3293, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
# Here you can see the overlap.

print(encoded_dict['input_ids'])
print(encoded_dict['overflowing_tokens'])

[0, 35378, 2685, 5, 11249, 621, 398, 32, 31901, 10, 26267, 5155, 5, 3293, 2]
[5155, 5, 3293, 83, 10, 3034, 32]


Because stride is set to 3, three tokens (5155, 5, 3293) from the end of the input_ids (excl. special token 2) are added to the front of the overflowing_tokens list. This creates an overlap between the two lists.

| <a id='Section_2'></a>

# Section 2

In this section we will load the competition train and test data. We will also create 5 folds that can be used for cross validation.

| <a id='Load_the_Data'></a>

## 2.1. Load the Data

In [ ]:
#load the data "train.csv" and "test.csv" you can find here : https://www.kaggle.com/code/naim99/basics-of-bert-and-xlm-roberta-pytorch/input



In [ ]:
# Load the training data.

path = 'train.csv'
df_train = pd.read_csv(path)

print(df_train.shape)

df_train.head()

(12120, 6)


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


In [ ]:
# Load the test data.

path = 'test.csv'
df_test = pd.read_csv(path)

print(df_test.shape)

df_test.head()

(5195, 5)


,id,premise,hypothesis,lang_abv,language
0,c6d58c3f69,بکس، کیسی، راہیل، یسعیاہ، کیلی، کیلی، اور کولم...,"کیسی کے لئے کوئی یادگار نہیں ہوگا, کولمین ہائی...",ur,Urdu
1,cefcc82292,هذا هو ما تم نصحنا به.,عندما يتم إخبارهم بما يجب عليهم فعله ، فشلت ال...,ar,Arabic
2,e98005252c,et cela est en grande partie dû au fait que le...,Les mères se droguent.,fr,French
3,58518c10ba,与城市及其他公民及社区组织代表就IMA的艺术发展进行对话&amp,IMA与其他组织合作，因为它们都依靠共享资金。,zh,Chinese
4,c32b0d16df,Она все еще была там.,"Мы думали, что она ушла, однако, она осталась.",ru,Russian


| <a id='Create_5_Folds'></a>

## 2.2. Create 5 Folds

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold

# shuffle
df = shuffle(df_train)

# initialize kfold
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1024)

# for stratification
y = df['label']

# Note:
# Each fold is a tuple ([train_index_values], [val_index_values])
# fold_0, fold_1, fold_2, fold_3, fold_5 = kf.split(df, y)

# Put the folds into a list. This is a list of tuples.
fold_list = list(kf.split(df, y))

train_df_list = []
val_df_list = []

for i, fold in enumerate(fold_list):

    # map the train and val index values to dataframe rows
    df_train = df[df.index.isin(fold[0])]
    df_val = df[df.index.isin(fold[1])]

    train_df_list.append(df_train)
    val_df_list.append(df_val)



print(len(train_df_list))
print(len(val_df_list))

5
5


In [ ]:
# Display one train fold

df_train = train_df_list[0]

df_train.head()

,id,premise,hypothesis,lang_abv,language,label
2306,64da9a705c,Each room was outfitted with a leather sofa an...,Students received housing with sofas and beds ...,en,English,0
6237,daabd0c7eb,"Also downtown is the Flower Market, on Wall an...",Fresh-cut flowers available at the market rang...,en,English,1
4517,90a1a7cd35,I did so.,I didn't.,en,English,2
7098,6bdd3a0025,Ο Mihdhar διαμαρτυρήθηκε για τη ζωή στις Ηνωμέ...,Ο Mihdhar διαμαρτυρήθηκε σε αρκετούς κοντινούς...,el,Greek,1
396,e0a29d673a,trying to keep grass alive during a summer on ...,There was no cost in keeping the grass alive i...,en,English,2


In [ ]:
# Display one val fold

df_val = val_df_list[0]

df_val.head()

,id,premise,hypothesis,lang_abv,language,label
10637,c4ff2a3e4c,Via di Ripetta不知不觉地融入了Via della Scrofa的“Street...,Via della Scrofa以另一件雕塑作品为名。,zh,Chinese,0
9870,22b39d8c51,"It's a great novelty, but very expensive.",Some find the experience isn't worth the price.,en,English,1
3785,c2beb55082,"Нашите най-агресивни респонденти, развълнувани...",Хората изобщо не отговаряха.,bg,Bulgarian,2
8165,469b0c268f,الوقت اللازم لإكمال هذه المرحلة من مشروع التنف...,يستغرق تنفيذ القواعد الجديدة 17 شهرًا.,ar,Arabic,1
1036,408db55a89,Since the rules were issued as interim rules a...,The rules were issued as interim rules and not...,en,English,0
